<a href="https://colab.research.google.com/github/diegobautista1012-lang/optimizador-rutas/blob/main/optimizador_de_rutas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
puntos = [
    ("deposito",         19.63627289905093, -99.200432555112),
    ("parada1",             19.64454081566007, -99.20515088504385),
    ("parada2",          19.63577572935039, -99.23944300946822),
    ("parada3", 19.65901738679457, -99.21075938822268),
    ("parada4",           19.64642486258566, -99.18887930812824),
    ("parada5",       19.57240666056739, -99.22000711272474),
    ("parada6",         19.59779939115131, -99.18847457071712),
    ("parada7",           19.49237842110952, -99.23731457525035),
    ("parada8",            19.43112350211384, -99.18824119826257),
]

print("Total de puntos:", len(puntos))
print("Primero:", puntos[0])

Total de puntos: 9
Primero: ('deposito', 19.63627289905093, -99.200432555112)


In [2]:
import math

def haversine(p1, p2):
    R = 6371.0
    lat1 = math.radians(p1[1])
    lon1 = math.radians(p1[2])
    lat2 = math.radians(p2[1])
    lon2 = math.radians(p2[2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

    return R * c
    print(haversine(puntos[0], puntos[1]))
print(haversine(puntos[0], puntos[8]))

22.847320826547268


In [3]:
D = []

for i in range(len(puntos)):
    fila = []
    for j in range(len(puntos)):
        fila.append(haversine(puntos[i], puntos[j]))
    D.append(fila)

print(len(D))
print(len(D[0]))
print(D[0][0])
print(D[0][8])
print(D[3][5] == D[5][3])

9
9
0.0
22.847320826547268
True


In [4]:
  actual = 0
  ruta = [0]
  total = 0.0

  while len(ruta) < len(puntos):
    mejor = None
    mejor_dist = float('inf')

    for j in range(len(puntos)):
      if j in ruta:
        continue
      if D[actual][j]  < mejor_dist:
        mejor_dist = D[actual][j]
        mejor = j

    total = total + mejor_dist
    ruta.append(mejor)
    actual = mejor

  total = total + D[actual][0]
  ruta.append(0)

  print(len(ruta))
  print(ruta[0], ruta[-1])
  print(sorted(ruta))
  print(total)
  print("->".join(puntos[i][0] for i in ruta))
  print("->".join(str(i) for i in ruta))



10
0 0
[0, 0, 1, 2, 3, 4, 5, 6, 7, 8]
69.8596765125682
deposito->club->centro comercial->planta->hospital->casa amigo->escuela->parque->museo->deposito
0->1->3->4->6->5->2->7->8->0


In [7]:
import itertools

mejor_ruta = None
mejor_total = float('inf')

for perm in itertools.permutations(range(1, 9)):
    d = D[0][perm[0]]

    for k in range(len(perm) - 1):
      d = d + D[perm[k]][perm[k+1]]

    d = d + D[perm[-1]][0]

    if d < mejor_total:
        mejor_total = d
        mejor_ruta = perm


print(mejor_total)
print(" -> ".join(puntos[i][0] for i in (0,) + mejor_ruta + (0,)))
print("Ruta voraz:  ", round(total, 2), "km")
print("Ruta óptima: ", round(mejor_total, 2), "km")
print("Mejora:      ", round((total - mejor_total) / total * 100, 1), "%")

56.99101539977727
deposito -> hospital -> museo -> parque -> casa amigo -> escuela -> centro comercial -> club -> planta -> deposito
Ruta voraz:   69.86 km
Ruta óptima:  56.99 km
Mejora:       18.4 %


# Optimizador de rutas de reparto
Calcular la ruta más corta para repartir en 8 puntos, y compararla contra la forma intuitiva de planearla.

## Resultado
-Ruta secuencial (ir siempre al punto más cercano)	= 69.86 km
-Ruta óptima	= 56.99 km

18.4% menos kilómetros. 12.87 km ahorrados en un solo reparto.

El óptimo está verificado contra 40,320 rutas posibles,  no es una estimación, es la ruta más corta que existe para esos puntos.

## El problema
Si se tiene que repartir en 8 lugares diferente y se visita en orden que parece lógico, lo más probable es que el recorrido sea más largo de lo necesario

La trampa es que pensar: "voy al que esté más cerca" se siente correcto en cada paso y falla en el conjunto. Al dejar los puntos lejanos para el final, obliga a un viaje largo de regreso. En esta ruta, ese último tramo mide 22.85 km: un tercio del recorrido completo en una sola línea.

La ruta óptima hace lo contrario. Sale primero hacia el grupo lejano y regresa pasando por el cercano de camino a casa. Sus primeros tramos son más largos, pero elimina el viaje final.

## Cómo funciona
Datos: Un depósito y 8 paradas reales, con coordenadas tomadas de Google Maps.

Distancias (Haversine): Calcula la distancia sobre la superficie de la esfera. No usé el teorema de Pitágoras sobre las coordenadas porque un grado de longitud no mide lo mismo en todas las latitudes: aquí son unos 105 km y en el ecuador 111.

Matriz de distancias: Las 81 distancias se calculan una sola vez y se guardan en una tabla de 9x9. El algoritmo consulta la tabla en lugar de recalcular senos y raíces en cada paso, y queda independiente de cómo se midan las distancias.

Dos algoritmos: El primero es una ruta voraz: siempre al punto más cercano. Da la línea base. El segundo recorre todas las rutas posibles con itertools y encuentra la ruta más óptima

## Cómo correrlo
Abre el notebook en Google Colab y corre las celdas de arriba a abajo. No hay nada que instalar: solo usa math e itertools, que vienen con Python.

Para usar otros puntos, cambia la lista puntos por tus propias coordenadas. Google Maps las da con clic derecho sobre el mapa.

## Limitaciones
Las distancias son en línea recta. Haversine mide de punto a punto sobre la esfera, no por calles. El manejo real es mayor, así que los dos números son optimistas. El porcentaje de mejora se sostiene porque las dos rutas se miden igual.

La búsqueda exhaustiva no escala. Con 8 paradas son 40,320 rutas y tarda menos de un segundo. Con 12 paradas serían 479 millones. Para más puntos hace falta un solver de optimización.

el programa solo toma en cuenta un solo vehículo, sin restricciones. No hay capacidad de carga, ni ventanas de horario, ni varios camiones.

## Qué sigue
-v2: capacidad del camión, múltiples vehículos, y función objetivo en pesos (combustible + horas + costo fijo por camión).
-v3: distancias reales por calle, ventanas de horario por parada, y la ruta dibujada sobre un mapa.